# House Prices — Advanced Regression Techniques

End-to-end walkthrough for Kaggle's Ames Housing competition. Download `train.csv` and `test.csv` from the competition Data tab and place them in `../data/`.

## 1. Imports and load data
We will predict `SalePrice`. The competition metric is RMSE in log-price space, so we model `log1p(SalePrice)`. 

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = Path('../data')
train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(train.shape, test.shape)
train.head()

## 2. Understand the target and missing values
Use the training data only for target analysis. Missing values are not automatically errors: for example, a missing garage-related value can mean the house has no garage.

In [ ]:
train['SalePrice'].describe()


In [ ]:
sns.histplot(train['SalePrice'], kde=True)
plt.title('SalePrice distribution')
plt.show()
sns.histplot(np.log1p(train['SalePrice']), kde=True)
plt.title('log1p(SalePrice) distribution')
plt.show()

In [ ]:
missing = train.isna().mean().sort_values(ascending=False)
missing[missing > 0].head(25)

## 3. Feature engineering
Create total area, bathroom count, house age, remodel age, porch area, and indicator features. The same transformations are applied to train and test.

In [ ]:
def add_features(df):
    df = df.copy()
    df['TotalSF'] = df[['TotalBsmtSF', '1stFlrSF', '2ndFlrSF']].fillna(0).sum(axis=1)
    df['TotalBathrooms'] = (df['FullBath'].fillna(0) + .5*df['HalfBath'].fillna(0) + df['BsmtFullBath'].fillna(0) + .5*df['BsmtHalfBath'].fillna(0))
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['RemodAge'] = df['YrSold'] - df['YearRemodAdd']
    df['TotalPorchSF'] = df[['OpenPorchSF','EnclosedPorch','3SsnPorch','ScreenPorch']].fillna(0).sum(axis=1)
    df['HasGarage'] = (df['GarageArea'].fillna(0) > 0).astype(int)
    df['HasBasement'] = (df['TotalBsmtSF'].fillna(0) > 0).astype(int)
    df['HasPool'] = (df['PoolArea'].fillna(0) > 0).astype(int)
    return df

X = add_features(train.drop(columns=['SalePrice','Id']))
X_test = add_features(test.drop(columns=['Id'])).reindex(columns=X.columns)
y = np.log1p(train['SalePrice'])
X.shape, X_test.shape

## 4. Preprocessing and validation
All imputing, scaling, and one-hot encoding happen inside a Pipeline. This prevents information from validation folds leaking into preprocessing.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, GradientBoostingRegressor

num_cols = X.select_dtypes(include='number').columns
cat_cols = X.select_dtypes(exclude='number').columns
preprocessor = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median', add_indicator=True)), ('scale', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', OneHotEncoder(handle_unknown='ignore'))]), cat_cols)
])

models = {
    'Ridge': Ridge(alpha=15),
    'Lasso': Lasso(alpha=0.0005, max_iter=20000),
    'ElasticNet': ElasticNet(alpha=0.0005, l1_ratio=.9, max_iter=20000),
    'RandomForest': RandomForestRegressor(n_estimators=500, max_features=.7, random_state=42, n_jobs=-1),
    'ExtraTrees': ExtraTreesRegressor(n_estimators=700, max_features=.8, random_state=42, n_jobs=-1),
    'GradientBoosting': GradientBoostingRegressor(n_estimators=1800, learning_rate=.03, max_depth=2, max_features=.8, loss='huber', random_state=42)
}
cv = KFold(n_splits=5, shuffle=True, random_state=42)


## 5. Compare models
The scoring function is negative RMSE because scikit-learn follows a convention where larger scores are better. We multiply by -1 to display positive RMSE.

In [ ]:
scores = {}
pipelines = {}
for name, model in models.items():
    pipe = Pipeline([('prep', preprocessor), ('model', model)])
    cv_scores = -cross_val_score(pipe, X, y, cv=cv, scoring='neg_root_mean_squared_error', n_jobs=1)
    scores[name] = (cv_scores.mean(), cv_scores.std())
    pipelines[name] = pipe
    print(f'{name:18s}: {cv_scores.mean():.5f} +/- {cv_scores.std():.5f}')

pd.DataFrame(scores, index=['mean_log_RMSE','std']).T.sort_values('mean_log_RMSE')

## 6. Fit top models and create submission
We blend the three best cross-validation models equally as a simple baseline. You can compare this with the single best model. Predictions are transformed back from log scale using `expm1`.

In [ ]:
ranking = sorted(scores, key=lambda k: scores[k][0])
top_models = ranking[:3]
print('Blending:', top_models)

preds = []
for name in top_models:
    pipelines[name].fit(X, y)
    preds.append(pipelines[name].predict(X_test))

pred_log = np.mean(np.column_stack(preds), axis=1)
sale_prices = np.maximum(np.expm1(pred_log), 1)
submission = pd.DataFrame({'Id': test['Id'], 'SalePrice': sale_prices})
submission.to_csv('../outputs_submission.csv', index=False)
submission.head()

## 7. Validate the submission file
Before uploading, verify row count, column names, missing predictions, and positive prices.

In [ ]:
assert list(submission.columns) == ['Id', 'SalePrice']
assert len(submission) == len(test)
assert submission['SalePrice'].notna().all()
assert (submission['SalePrice'] > 0).all()
print('Submission file looks valid:', submission.shape)